# Webinar 4 — Pipeline & Cross Validation
**Dataset: Titanic**

---
1. Cross Validation
2. Pipeline
---

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score

df = sns.load_dataset('titanic')[['survived', 'pclass', 'sex', 'fare']].dropna()


Train: (712, 3) | Test: (179, 3)


---
## 1. Cross Validation

Cuando evaluamos un modelo con un solo train/test split, el resultado depende de *cómo* cayó ese split — puede ser suerte.

**Cross Validation** repite el proceso `k` veces con distintas particiones y promedia los resultados.

```
Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]
```

El árbol de decisión es un modelo que **puede memorizar** los datos de entrenamiento si lo dejamos crecer sin límite. Con CV vamos a ver ese efecto claramente.

In [ ]:
#Revisa los datos y las distribuciones del mismo

,survived,pclass,sex,fare
0,0,3,male,7.2500
1,1,1,female,71.2833
2,1,3,female,7.9250
3,1,1,female,53.1000
4,0,3,male,8.0500


In [ ]:
#Realiza la separación de los datos

In [ ]:
# Árbol sin límite — memoriza el train (overfitting)
arbol_libre = #Monta un arbol de desición


#astype sintaxix
X_encoded = #Realiza el encoding de las variables categóricas

scores_libre = cross_val_score(arbol_libre, X_encoded, y_train, cv=5, scoring='accuracy')
print("Sin límite de profundidad:")
print("  Accuracy por fold:", scores_libre.round(3))
print(f"  Promedio: {scores_libre.mean():.3f} ± {scores_libre.std():.3f}")


#Revisa si el modelo está sobreajustado o no

Sin límite de profundidad:
  Accuracy por fold: [0.811 0.783 0.81  0.796 0.824]
  Promedio: 0.805 ± 0.014


In [ ]:
# ¿Cuál es la mejor profundidad? CV nos lo dice
for depth in [1, 2, 3, 5, 7 , 10, None]:
    model_test = #Monta un arbol de decisión con la profundidad dada
    scores_test = cross_val_score(model_test, X_encoded, y_train, cv=5, scoring='accuracy') 
    print(f"Con max_depth={depth}:")
    #print("  Accuracy por fold:", scores_test.round(3))
    print(f"  Promedio: {scores_test.mean():.3f} ± {scores_test.std():.3f}")

Con max_depth=1:
  Promedio: 0.788 ± 0.029
Con max_depth=2:
  Promedio: 0.784 ± 0.025
Con max_depth=3:
  Promedio: 0.803 ± 0.020
Con max_depth=5:
  Promedio: 0.793 ± 0.034
Con max_depth=7:
  Promedio: 0.809 ± 0.019
Con max_depth=10:
  Promedio: 0.805 ± 0.019
Con max_depth=None:
  Promedio: 0.805 ± 0.014


In [ ]:
#Evaluamos el modelo con la mejor profundidad en el test set y train set

              precision    recall  f1-score   support

           0       0.81      0.90      0.85       105
           1       0.83      0.70      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.81       179

0.8156424581005587


El **promedio** nos dice qué tan bueno es el modelo.  
La **desviación** nos dice qué tan estable es entre distintos splits.

Un modelo con `0.80 ± 0.01` es mucho más confiable que uno con `0.80 ± 0.08`.

---
## 2. Pipeline

Un Pipeline encadena pasos de preprocesamiento + modelo en un solo objeto.

**Sin Pipeline** el flujo es manual y fácil de romper:

In [ ]:
# ❌ Sin Pipeline
from sklearn.preprocessing import StandardScaler

"""
paso a paso el código
1. X, y train test
2. transformó : numericas --> StandardScaler categoricas --> OneHotEncoder 
3. entreno el modelo con los datos transformados
4. transformo el test con las mismas transformaciones del train
5. predigo con el modelo entrenado y los datos transformados del test
"""

# Entrenemos un modelo con max_depth=3 sin usar Pipeline, para ver lo que hay que hacer manualmente:

print(X_train.head())
scaler = StandardScaler() # operación matemática que estandariza las columnas numéricas (media=0, std=1)
X_train_manual = X_train.copy()
X_test_manual  = X_test.copy()

# Convertir de x categorical a numérica manualmente
X_train_manual['sex'] = (X_train_manual['sex'] == 'male').astype(int)
#Escalar las columnas numéricas
X_train_manual[['fare', 'pclass']] = scaler.fit_transform(X_train_manual[['fare', 'pclass']])

print(X_train_manual.head())

#ahora si entrenamos el árbol con los datos manualmente procesados
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train_manual, y_train)

X_test_manual['sex']  = (X_test_manual['sex']  == 'male').astype(int)
X_test_manual[['fare', 'pclass']]  = scaler.transform(X_test_manual[['fare', 'pclass']])

y_pred = arbol.predict(X_test_manual)

print(classification_report(y_test, y_pred))

     pclass     sex     fare
331       1    male  28.5000
733       2    male  13.0000
382       3    male   7.9250
704       3    male   7.8542
813       3  female  31.2750
       pclass  sex      fare
331 -1.614136    1 -0.078684
733 -0.400551    1 -0.377145
382  0.813034    1 -0.474867
704  0.813034    1 -0.476230
813  0.813034    0 -0.025249
              precision    recall  f1-score   support

           0       0.79      0.88      0.83       105
           1       0.79      0.68      0.73        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



**Con Pipeline** todo queda encadenado y el fit/transform se maneja automáticamente:

In [ ]:
# ✅ Con Pipeline

"""
paso a paso el código
1. X, y train test --> Manual


2. transformó : numericas --> StandardScaler categoricas --> OneHotEncoder  : transformador

3. entreno el modelo con los datos transformados
4. transformo el test con las mismas transformaciones del train
5. predigo con el modelo entrenado y los datos transformados del test
"""

transformador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['fare', 'pclass']),
    ('cat', OneHotEncoder(drop='first'), ['sex']),
])

pipe = Pipeline(steps=[
    ('preprocessor', transformador)  ,
    ('model', DecisionTreeClassifier(max_depth=3, random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.88      0.83       105
           1       0.79      0.68      0.73        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



### Cross Validation con Pipeline

La combinación más poderosa: el pipeline garantiza que el preprocesamiento
ocurra **dentro de cada fold**, sin data leakage.

In [40]:
scores_pipe = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

print("Accuracy por fold:", scores_pipe.round(3))
print(f"Promedio: {scores_pipe.mean():.3f} ± {scores_pipe.std():.3f}")

Accuracy por fold: [0.818 0.832 0.789 0.775 0.803]
Promedio: 0.803 ± 0.020


---
## 3. GridSearchCV

Antes hacíamos esto a mano para encontrar el mejor `max_depth`:
```python
for depth in [1, 2, 3, 5, 10]:
    ...
```

**GridSearchCV** hace exactamente eso pero de forma automática, probando todas las combinaciones con Cross Validation incluido.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__max_depth': [1, 2, 3, 5, 7, 10],
    "model__min_samples_split": [2, 5, 10],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("Mejor max_depth:", grid.best_params_)
print(f"Mejor accuracy en CV: {grid.best_score_:.3f}")

Mejor max_depth: {'model__max_depth': 7, 'model__min_samples_split': 2}
Mejor accuracy en CV: 0.809


---
## Resumen

| Concepto | Para qué sirve |
|---|---|
| **Cross Validation** | Evaluar el modelo de forma robusta, detectar overfitting |
| **Pipeline** | Encadenar preprocesamiento + modelo en un solo objeto |
| **Juntos** | CV con Pipeline garantiza que no haya data leakage en ningún fold |